# **Clasificador de imagenes usando redes neuronales**

En este tutorial, entrenará una red neuronal totalmente conectada para realizar la clasificación de imágenes en dientes de león y hierba. Utilizaremos TensorFlow/Keras para construir nuestra red neuronal y conjunto de datos de entrenamiento/validación. Primero, descarguemos los datos de capacitación y validación


**Extensión opcional:** reutiliza las mismas imágenes del cuaderno 4 con una sola capa oculta. Compare profundidad, tiempo y validación; no repita la lección de descarga como un tema nuevo.


### Creación del conjunto de datos de imágenes

El set de datos corresponden a una colección de fotos desde internet

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import os
import zipfile
data_dir = Path('datos_imagenes').resolve()
data_dir.mkdir(exist_ok=True)
base_url = 'https://raw.githubusercontent.com/btphan95/simple-cnn/master/'
train_zip = data_dir / 'train.zip'
if not train_zip.exists():
    urlretrieve(base_url + 'train.zip', train_zip)


In [ ]:
valid_zip = data_dir / 'valid.zip'
if not valid_zip.exists():
    urlretrieve(base_url + 'valid.zip', valid_zip)


El siguiente código python usará la biblioteca del sistema operativo para usar las bibliotecas del sistema operativo, lo que le dará acceso al sistema de archivos y a la biblioteca zipfile que le permitirá descomprimir los datos.

In [ ]:
# Comprobar que ningún miembro del ZIP sale del directorio de datos.
for archive in (train_zip, valid_zip):
    with zipfile.ZipFile(archive) as z:
        for member in z.infolist():
            destination = (data_dir / member.filename).resolve()
            if not destination.is_relative_to(data_dir):
                raise ValueError('Ruta no permitida dentro del ZIP')
        z.extractall(data_dir)


El contenido de los .zip se extrae al directorio '/tmp/train', que a su vez contiene los subdirectorios 'diente de león' y 'hierba'.

Lo que verá más adelante es que usaremos la clase [ImageGenerator](https://www.tensorflow.org/api_docs/python/tf/keras/preprocessing/image/ImageDataGenerator) para crear automáticamente nuestro conjunto de datos a partir de este directorio de trenes, utilizando los subdirectorios para las clases.

Definamos cada uno de estos subdirectorios:

In [ ]:
# Directorio con imagenes de entrenamiento con fotos de dientes de león
train_dandelion_dir = str(data_dir / 'train/dandelion')

# Directorio con imagenes de entrenamiento con fotos de hierba
train_grass_dir = str(data_dir / 'train/grass')

# Directorio con imagenes de validación con fotos de dientes de león
valid_dandelion_dir = str(data_dir / 'valid/dandelion')

# Directorio con imagenes de validación con fotos de hierba
valid_grass_dir = str(data_dir / 'valid/grass')

Ahora, veamos cómo se ven los nombres de archivo en los directorios de entrenamiento 'diente de león' y 'hierba':

In [ ]:
train_dandelion_names = os.listdir(train_dandelion_dir)
print(train_dandelion_names[:10])

train_grass_names = os.listdir(train_grass_dir)
print(train_grass_names[:10])

validation_dandelion_hames = os.listdir(valid_dandelion_dir)
print(validation_dandelion_hames[:10])

validation_grass_names = os.listdir(valid_grass_dir)
print(validation_grass_names[:10])

Averigüemos el número total de imágenes de diente de león y hierba en los directorios:

In [ ]:
print('total training dandelion images:', len(os.listdir(train_dandelion_dir)))
print('total training grass images:', len(os.listdir(train_grass_dir)))
print('total validation dandelion images:', len(os.listdir(valid_dandelion_dir)))
print('total validation grass images:', len(os.listdir(valid_grass_dir)))

Ahora echemos un vistazo a algunas imágenes para tener una mejor idea de cómo se ven. Primero, configure los parámetros de matplot:

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Para crear un marco con 4 x 4 fotos
nrows = 4
ncols = 4

# iterador de la imágenes
pic_index = 0

Ahora, muestre un lote de 8 imágenes de diente de león y 8 de hierba. Puede volver a ejecutar la celda para ver un lote nuevo cada vez:

In [ ]:
# Set up matplotlib fig, and size it to fit 4x4 pics
fig = plt.gcf()
fig.set_size_inches(ncols * 4, nrows * 4)

pic_index += 8
next_dandelion_pic = [os.path.join(train_dandelion_dir, fname) 
                for fname in train_dandelion_names[pic_index-8:pic_index]]
next_grass_pic = [os.path.join(train_grass_dir, fname) 
                for fname in train_grass_names[pic_index-8:pic_index]]

for i, img_path in enumerate(next_dandelion_pic + next_grass_pic):
  # Set up subplot; subplot indices start at 1
  sp = plt.subplot(nrows, ncols, i + 1)
  sp.axis('Off') # Don't show axes (or gridlines)

  img = mpimg.imread(img_path)
  plt.imshow(img)

plt.show()


#### Preprocesamiento de datos

Ahora, usemos la clase 'keras.preprocessing.image.ImageDataGenerator' para crear nuestro conjunto de datos de tren y validación y normalizar nuestros datos. 

Es importante normalizar nuestros datos porque los datos que entran en nuestra red densa mejoran su rendimiento general. Usaremos el parámetro 'reescalar' para escalar los valores de píxeles de nuestra imagen de [0, 255] a [0,1].

En cada generador, especificamos el directorio de origen de nuestras imágenes, las clases, el tamaño de la imagen de entrada, el tamaño del lote (cuántas imágenes procesar a la vez) y el modo de clase.

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# All images will be rescaled by 1./255
train_datagen = ImageDataGenerator(rescale=1/255)
validation_datagen = ImageDataGenerator(rescale=1/255)

# Flow training images in batches of 120 using train_datagen generator
train_generator = train_datagen.flow_from_directory(
        str(data_dir / 'train'),  # This is the source directory for training images
        classes = ['dandelion', 'grass'],
        target_size=(200, 200),  # All images will be resized to 200x200
        batch_size=120,
        # Use binary labels
        class_mode='binary', seed=42)

# Flow validation images in batches of 19 using valid_datagen generator
validation_generator = validation_datagen.flow_from_directory(
        str(data_dir / 'valid'),  # This is the source directory for training images
        classes = ['dandelion', 'grass'],
        target_size=(200, 200),  # All images will be resized to 200x200
        batch_size=19,
        # Use binary labels
        class_mode='binary',
        shuffle=False)

## Construyendo el modelo desde cero

Pero antes de continuar, comencemos a definir el modelo:

El paso 1 será importar tensorflow.

In [ ]:
import tensorflow as tf
tf.keras.utils.set_random_seed(42)
import numpy as np
from itertools import cycle

from sklearn import svm, datasets
from sklearn.metrics import roc_curve, auc
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
from sklearn.multiclass import OneVsRestClassifier
# Se eliminó una importación obsoleta que no se utilizaba.
from sklearn.metrics import roc_auc_score

Luego agreguemos una capa aplanada que aplana la imagen de entrada, que luego alimenta a la siguiente capa, una capa densa o una capa completamente conectada, con 128 unidades ocultas. Finalmente, debido a que nuestro objetivo es realizar la clasificación binaria, nuestra capa final será un sigmoide, de modo que la salida de nuestra red será un único escalar entre 0 y 1, codificando la probabilidad de que la imagen actual sea de clase 1 (clase 1 siendo hierba y clase 0 siendo diente de león).

In [ ]:
model = tf.keras.models.Sequential([tf.keras.layers.Flatten(input_shape = (200,200,3)), 
                                    tf.keras.layers.Dense(128, activation=tf.nn.relu), 
                                    tf.keras.layers.Dense(1, activation=tf.nn.sigmoid)])

The model.summary() method call prints a summary of the NN 

In [ ]:
model.summary()

La red es densa, no convolucional. Flatten convierte los 200 × 200 × 3 valores de cada imagen en un vector; las capas Dense aprenden combinaciones de esos valores. No hay convoluciones ni max pooling en esta arquitectura.

A continuación, configuraremos las especificaciones para el entrenamiento de modelos. Entrenaremos nuestro modelo con la pérdida de 'binary_crossentropy'. Utilizaremos el optimizador 'Adam'. [Adán] (https://wikipedia.org/wiki/Stochastic_gradient_descent#Adam) es un algoritmo de optimización sensato porque automatiza el ajuste de la tasa de aprendizaje para nosotros (alternativamente, también podríamos usar [RMSProp] (https://wikipedia.org/wiki/Stochastic_gradient_descent#RMSProp) o [Adagrad] (https://developers.google.com/machine-learning / glossary / #AdaGrad) para resultados similares). Agregaremos precisión a las "métricas" para que el modelo monitoree la precisión durante el entrenamiento.

In [ ]:
model.compile(optimizer = tf.optimizers.Adam(),
              loss = 'binary_crossentropy',
              metrics=['accuracy'])

### Entrenamiento
Se recorren todos los lotes disponibles durante 15 épocas. Compare pérdida y exactitud de entrenamiento y validación; que una supere a la otra no demuestra por sí solo un buen ajuste.

In [ ]:
history = model.fit(train_generator, epochs=15, verbose=1,
                    validation_data=validation_generator)


## Predicción de una imagen
Se usa una imagen de validación como demostración. Sustituya imagen_path por la ruta de otra imagen para practicar. La predicción individual no equivale a una evaluación independiente del modelo.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from tensorflow.keras.utils import load_img, img_to_array
imagen_path = Path(validation_generator.filepaths[0])
img = load_img(imagen_path, target_size=(200, 200))
x = img_to_array(img).astype('float32') / 255.0
plt.imshow(x)
plt.axis('off')
plt.show()
classes = model.predict(np.expand_dims(x, axis=0), verbose=0)
prob_grass = float(classes[0, 0])
label = 'grass' if prob_grass >= 0.5 else 'dandelion'
print(imagen_path.name, label, f'P(grass)={prob_grass:.3f}')
